In [1]:
# ------------------------------------------------------------
# 📦 Imports & Configuration
# ------------------------------------------------------------
import os
import cv2
import time
import torch
import numpy as np
import pandas as pd
from collections import deque, defaultdict
from ultralytics import YOLO
from sklearn.metrics import average_precision_score

# ------------------------------------------------------------
# JSON STYLE CONFIG
# ------------------------------------------------------------
CONFIG = {
    # Input video + ground truth
    "VIDEO_PATH": "/home1/jtt_1/mtp/timesformer-dataset-test/Drone2/Morning/2.1.8.mp4",
    "GT_FILE": "/home1/jtt_1/mtp/timesformer-dataset-test/Drone2/Morning/2.1.8_bboxes.csv",

    # Models
    "YOLO_MODEL_PATH": "/home1/jtt_1/mtp/models-2/best.pt",
    "TSF_MODEL_PATH": "/home1/jtt_1/mtp/timesformer_base_patch16_224_k400",
    "TSF_CHECKPOINT": "/home1/jtt_1/mtp/models-2/best_timesformer_epoch68.pt",

    # Training CSV (for collecting all action classes)
    "GT_BBOX_CSV": "/home1/jtt_1/mtp/timesformer-person-dataset/train_person_labels.csv",

    # Hyperparameters
    "CLIP_LEN": 16,
    "NUM_FRAMES": 8,
    "TSF_INPUT_SIZE": 224,
    "IOU_THRESH": 0.5,

    # Detection settings
    "CONF_THRESH": 0.7,
    "OUT_OF_FRAME_FRACTION": 0.4,
    "REID_MEMORY_TIME": 4.0,
    "TOP_K": 2,

    # Test settings
    "TEST_MODE": True,
    "TEST_MAX_FRAMES": 300,

    # Device
    "DEVICE": "cuda" if torch.cuda.is_available() else "cpu",
    "DETECT_EVERY_N_FRAMES":1
    }

# ------------------------------------------------------------
# AUTO FIELDS (computed)
# ------------------------------------------------------------
CONFIG["VIDEO_NAME"] = os.path.splitext(os.path.basename(CONFIG["VIDEO_PATH"]))[0]
CONFIG["OUTPUT_DIR"] = f"/home1/jtt_1/mtp/outputs/{CONFIG['VIDEO_NAME']}"
os.makedirs(CONFIG["OUTPUT_DIR"], exist_ok=True)

CONFIG["RESULTS_CSV"] = f"{CONFIG['OUTPUT_DIR']}/{CONFIG['VIDEO_NAME']}_tracking.csv"

print("[INFO] VIDEO_NAME   :", CONFIG["VIDEO_NAME"])
print("[INFO] OUTPUT_DIR   :", CONFIG["OUTPUT_DIR"])
print("[INFO] RESULTS_CSV  :", CONFIG["RESULTS_CSV"])

# ------------------------------------------------------------
# Load unique ACTION_CLASSES from GT CSV
# ------------------------------------------------------------
df = pd.read_csv(CONFIG["GT_BBOX_CSV"])

unique_actions = set()
for acts in df["Actions"]:
    if pd.isna(acts):
        continue
    for a in str(acts).split(","):
        unique_actions.add(a.strip())

CONFIG["ACTION_CLASSES"] = sorted(unique_actions)
CONFIG["NUM_CLASSES"] = len(CONFIG["ACTION_CLASSES"])

print("[INFO] Loaded ACTION_CLASSES:", CONFIG["ACTION_CLASSES"])
print("[INFO] NUM_CLASSES:", CONFIG["NUM_CLASSES"])
print("[INFO] Device:", CONFIG["DEVICE"])


[INFO] VIDEO_NAME   : 2.1.8
[INFO] OUTPUT_DIR   : /home1/jtt_1/mtp/outputs/2.1.8
[INFO] RESULTS_CSV  : /home1/jtt_1/mtp/outputs/2.1.8/2.1.8_tracking.csv
[INFO] Loaded ACTION_CLASSES: ['Calling', 'Carrying', 'Drinking', 'Handshaking', 'Hugging', 'Lying', 'Pushing or Pulling', 'Reading', 'Running', 'Sitting', 'Standing', 'Walking', 'nan']
[INFO] NUM_CLASSES: 13
[INFO] Device: cuda


In [2]:
# ------------------------------------------------------------
# Load Ground Truth BBOX CSV
# ------------------------------------------------------------
GT_FILE = CONFIG["VIDEO_PATH"].replace(".mp4", "_bboxes.csv")

if not os.path.exists(GT_FILE):
    print(f"[ERROR] GT file not found: {GT_FILE}")
else:
    print(f"[INFO] Loading Ground Truth from: {GT_FILE}")

gt_df = pd.read_csv(GT_FILE)

# Organize GT per frame
gt_by_frame = defaultdict(list)

for idx, row in gt_df.iterrows():
    frame_id = int(row["frame"])
    gt_box = {
        "track_id": int(row["track_id"]),
        "bbox": [row["xmin"], row["ymin"], row["xmax"], row["ymax"]],
        "action": row["actions"]
    }
    gt_by_frame[frame_id].append(gt_box)

print(f"[INFO] Loaded GT for {len(gt_by_frame)} frames.")


[INFO] Loading Ground Truth from: /home1/jtt_1/mtp/timesformer-dataset-test/Drone2/Morning/2.1.8_bboxes.csv
[INFO] Loaded GT for 2227 frames.


In [3]:
cap = cv2.VideoCapture(CONFIG["VIDEO_PATH"])

if not cap.isOpened():
    print(f"[ERROR] Could not open video file: {CONFIG['VIDEO_PATH']}")
else:
    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    duration_sec = total_frames / fps if fps > 0 else 0

    print(f"[INFO] Video Name       : {CONFIG['VIDEO_NAME']}")
    print(f"[INFO] Total Frames     : {total_frames}")
    print(f"[INFO] FPS              : {fps:.2f}")
    print(f"[INFO] Duration (secs)  : {duration_sec:.2f}")
    print(f"[INFO] Duration (mins)  : {duration_sec/60:.2f}")

cap.release()

[INFO] Video Name       : 2.1.8
[INFO] Total Frames     : 2227
[INFO] FPS              : 20.00
[INFO] Duration (secs)  : 111.35
[INFO] Duration (mins)  : 1.86


In [4]:
# ------------------------------------------------------------
# 🧠 LOAD MODELS (FINAL WORKING VERSION)
# ------------------------------------------------------------
print("[INFO] Loading YOLO model...")
yolo = YOLO(CONFIG["YOLO_MODEL_PATH"])
print("[INFO] YOLO loaded successfully.")


# ------------------------------------------------------------
# Load TimeSformer with correct number of action classes
# ------------------------------------------------------------
print("[INFO] Loading TimeSformer model...")

from transformers import AutoImageProcessor, TimesformerForVideoClassification
import torch.nn as nn

# Load action class list
action_labels = CONFIG["ACTION_CLASSES"]
num_actions = CONFIG["NUM_CLASSES"]

print(f"[INFO] TimeSformer will use {num_actions} action classes:")
print(action_labels)


# 1. Load processor from BASE MODEL DIRECTORY
processor = AutoImageProcessor.from_pretrained(CONFIG["TSF_MODEL_PATH"])


# 2. Load base model architecture
tsf_model = TimesformerForVideoClassification.from_pretrained(CONFIG["TSF_MODEL_PATH"])


# 3. Replace classification head
tsf_model.classifier = nn.Linear(tsf_model.classifier.in_features, num_actions)


# 4. Load your fine-tuned checkpoint (.pt file)
state_dict = torch.load(CONFIG["TSF_CHECKPOINT"], map_location=CONFIG["DEVICE"])
tsf_model.load_state_dict(state_dict, strict=True)


# 5. Move model to device
tsf_model = tsf_model.to(CONFIG["DEVICE"]).eval()

print("[INFO] TimeSformer loaded successfully.")
print("-------------------------------------")


[INFO] Loading YOLO model...
[INFO] YOLO loaded successfully.
[INFO] Loading TimeSformer model...


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


[INFO] TimeSformer will use 13 action classes:
['Calling', 'Carrying', 'Drinking', 'Handshaking', 'Hugging', 'Lying', 'Pushing or Pulling', 'Reading', 'Running', 'Sitting', 'Standing', 'Walking', 'nan']
[INFO] TimeSformer loaded successfully.
-------------------------------------


In [5]:
import numpy as np
from scipy.optimize import linear_sum_assignment
import cv2
import time

# ----------------------
# Helper functions
# ----------------------
def bbox_to_xywh(b):
    x1, y1, x2, y2 = b
    return [(x1 + x2) / 2.0, (y1 + y2) / 2.0, x2 - x1, y2 - y1]

def clamp(v, a, b):
    return max(a, min(b, v))

def crop_box(image, bbox, pad=0):
    h, w = image.shape[:2]
    x1, y1, x2, y2 = int(bbox[0]) - pad, int(bbox[1]) - pad, int(bbox[2]) + pad, int(bbox[3]) + pad
    x1, y1 = clamp(x1, 0, w-1), clamp(y1, 0, h-1)
    x2, y2 = clamp(x2, 0, w-1), clamp(y2, 0, h-1)
    if x2 <= x1 or y2 <= y1:
        return None
    return image[y1:y2, x1:x2]

def color_histogram_feature(image, bbox, size=(32, 32), bins=(16, 16, 8)):
    """Compute a normalized color histogram (HSV) for the bbox region."""
    crop = crop_box(image, bbox)
    if crop is None or crop.size == 0:
        # fallback: small zero vector
        return np.zeros(bins[0] + bins[1] + bins[2], dtype=np.float32)
    crop = cv2.resize(crop, size, interpolation=cv2.INTER_LINEAR)
    hsv = cv2.cvtColor(crop, cv2.COLOR_BGR2HSV)
    h_bins, s_bins, v_bins = bins
    hist_h = cv2.calcHist([hsv], [0], None, [h_bins], [0, 180]).flatten()
    hist_s = cv2.calcHist([hsv], [1], None, [s_bins], [0, 256]).flatten()
    hist_v = cv2.calcHist([hsv], [2], None, [v_bins], [0, 256]).flatten()
    hist = np.concatenate([hist_h, hist_s, hist_v]).astype(np.float32)
    if hist.sum() > 0:
        hist /= hist.sum()
    return hist

def cosine_sim(a, b):
    if a is None or b is None or a.sum()==0 or b.sum()==0:
        return 0.0
    na = np.linalg.norm(a)
    nb = np.linalg.norm(b)
    if na == 0 or nb == 0:
        return 0.0
    return float(np.dot(a, b) / (na * nb))


# ----------------------
# Particle filter class (unchanged core but exposed for predict/estimate)
# ----------------------
class Particle:
    def __init__(self, bbox, num_particles=50):
        self.num_particles = num_particles
        self.particles = self.initialize_particles(bbox)
        self.weights = np.ones(num_particles) / num_particles

    def initialize_particles(self, bbox):
        x1, y1, x2, y2 = bbox
        w, h = max(1, x2 - x1), max(1, y2 - y1)
        p = np.zeros((self.num_particles, 4))
        # sample around the box corners rather than absolute box for stability
        p[:, 0] = np.random.normal((x1 + x2) / 2.0 - w * 0.5, w * 0.05, self.num_particles)
        p[:, 1] = np.random.normal((y1 + y2) / 2.0 - h * 0.5, h * 0.05, self.num_particles)
        p[:, 2] = np.random.normal((x1 + x2) / 2.0 + w * 0.5, w * 0.05, self.num_particles)
        p[:, 3] = np.random.normal((y1 + y2) / 2.0 + h * 0.5, h * 0.05, self.num_particles)
        return p

    def predict(self, sigma=3.0):
        # smaller sigma for smoother motion assumptions
        self.particles += np.random.normal(0, sigma, self.particles.shape)

    def update(self, det):
        d = np.linalg.norm(self.particles - det, axis=1)
        # gaussian weight on L2 distance
        self.weights = np.exp(-d**2 / (2 * (30.0**2)))
        self.weights += 1e-300
        self.weights /= np.sum(self.weights)
        self.resample()

    def resample(self):
        idx = np.random.choice(self.num_particles, self.num_particles, p=self.weights)
        self.particles = self.particles[idx]
        self.weights = np.ones(self.num_particles) / self.num_particles

    def estimate(self):
        return np.average(self.particles, weights=self.weights, axis=0)


# ----------------------
# Track wrapper
# ----------------------
class ParticleTrack:
    def __init__(self, bbox, tid, feature=None, timestamp=None,
                 num_particles=50, confirm_hits=3, ema_alpha=0.2):
        self.id = tid
        self.pf = Particle(bbox, num_particles=num_particles)
        self.lost = 0
        self.age = 1
        self.hits = 1  # number of times associated
        self.confirm_hits = confirm_hits
        self.confirmed = (self.hits >= self.confirm_hits)
        self.last_box = bbox
        self.last_seen = timestamp if timestamp is not None else time.time()
        # appearance feature (vector) - if available, keep EMA average
        self.feature = feature.copy() if feature is not None else None
        self.ema_alpha = ema_alpha

    def predict(self):
        self.pf.predict()
        est = self.pf.estimate()
        # estimated box may be in particle coordinate form: [x1,y1,x2,y2]
        self.age += 1
        return est

    def update(self, bbox, feature=None, timestamp=None):
        self.pf.update(bbox)
        self.last_box = bbox
        self.last_seen = timestamp if timestamp is not None else time.time()
        self.lost = 0
        self.hits += 1
        if feature is not None:
            if self.feature is None:
                self.feature = feature.copy()
            else:
                # exponential moving average to adapt appearance slowly
                self.feature = (1.0 - self.ema_alpha) * self.feature + self.ema_alpha * feature
        self.confirmed = (self.hits >= self.confirm_hits)


# ----------------------
# Tracker main class
# ----------------------
class ParticleTracker:
    def __init__(self,
                 max_lost=30,
                 iou_weight=0.5,
                 dist_weight=0.2,
                 app_weight=0.3,
                 distance_decay=80.0,
                 min_confirm_hits=3,
                 reid_memory_time=3.0,
                 num_particles=50):
        """
        max_lost: frames after which a track is considered gone
        iou_weight, dist_weight, app_weight: weights for the combined matching cost (sum=1)
        distance_decay: used to convert center distance -> similarity
        min_confirm_hits: hits required to mark a track 'confirmed'
        reid_memory_time: seconds to keep recently lost tracks in memory for re-id
        """
        self.tracks = {}            # active tracks: id -> ParticleTrack
        self.next_id = 0
        self.max_lost = max_lost
        self.iou_w = iou_weight
        self.dist_w = dist_weight
        self.app_w = app_weight
        assert abs(self.iou_w + self.dist_w + self.app_w - 1.0) < 1e-6, "weights must sum to 1"
        self.distance_decay = distance_decay
        self.min_confirm_hits = min_confirm_hits
        self.reid_memory_time = reid_memory_time
        self.memory = {}            # id -> (last_box, feature, last_seen_time)
        self.num_particles = num_particles

    # ----------------------
    # geometry / similarity helpers
    # ----------------------
    @staticmethod
    def iou(b1, b2):
        x1, y1, x2, y2 = max(b1[0], b2[0]), max(b1[1], b2[1]), min(b1[2], b2[2]), min(b1[3], b2[3])
        w, h = max(0, x2 - x1), max(0, y2 - y1)
        inter = w * h
        area1 = max(0, (b1[2]-b1[0])*(b1[3]-b1[1]))
        area2 = max(0, (b2[2]-b2[0])*(b2[3]-b2[1]))
        union = area1 + area2 - inter
        return inter / union if union > 0 else 0.0

    @staticmethod
    def center_distance(b1, b2):
        c1 = ((b1[0]+b1[2])/2.0, (b1[1]+b1[3])/2.0)
        c2 = ((b2[0]+b2[2])/2.0, (b2[1]+b2[3])/2.0)
        return np.linalg.norm(np.array(c1) - np.array(c2))

    def appearance_similarity(self, f1, f2):
        # returns similarity in [0,1]
        if f1 is None or f2 is None:
            return 0.0
        return clamp(cosine_sim(f1, f2), 0.0, 1.0)

    # ----------------------
    # Matching
    # ----------------------
    def build_cost_matrix(self, track_boxes, track_features, detections, det_features):
        n_t = len(track_boxes)
        n_d = len(detections)
        if n_t == 0 or n_d == 0:
            return np.zeros((n_t, n_d), dtype=np.float32)

        cost = np.zeros((n_t, n_d), dtype=np.float32)
        for i, tb in enumerate(track_boxes):
            for j, db in enumerate(detections):
                iou_score = self.iou(tb, db)          # [0,1]
                dist = self.center_distance(tb, db)
                dist_score = np.exp(-dist / (self.distance_decay + 1e-9))  # [0,1]
                app_score = self.appearance_similarity(track_features[i], det_features[j])  # [0,1]
                # combined similarity
                sim = self.iou_w * iou_score + self.dist_w * dist_score + self.app_w * app_score
                # cost for hungarian is 1 - similarity (lower better)
                cost[i, j] = 1.0 - sim
        return cost

    # ----------------------
    # Public update: detections can be list of bboxes or list of dict {'bbox':..,'feature':..}
    # frame is required if detections are raw bboxes and we want to compute appearance features
    # ----------------------
    def update(self, detections, frame=None, timestamp=None):
        """
        detections: list of bboxes [x1,y1,x2,y2] OR list of dicts {'bbox':..., 'feature':...}
        frame: current video frame (BGR numpy array) - used to compute appearance features if needed
        returns: list of tuples (bbox, track_id, confirmed_bool)
        """
        timestamp = timestamp if timestamp is not None else time.time()

        # normalize detections to (bbox, feature) pairs
        det_bboxes = []
        det_features = []
        for det in detections:
            if isinstance(det, dict):
                bbox = det['bbox']
                feature = det.get('feature', None)
            else:
                bbox = det
                feature = None
            det_bboxes.append(bbox)
            if feature is None:
                # compute simple color histogram if frame is available, else None
                if frame is not None:
                    feature = color_histogram_feature(frame, bbox)
                else:
                    feature = None
            det_features.append(feature)

        # If no active tracks, create new tracks for all detections
        if not self.tracks:
            outputs = []
            for j, bbox in enumerate(det_bboxes):
                fid = self.next_id
                tr = ParticleTrack(bbox, fid, feature=det_features[j],
                                   timestamp=timestamp, num_particles=self.num_particles,
                                   confirm_hits=self.min_confirm_hits)
                tr.confirmed = tr.hits >= self.min_confirm_hits
                self.tracks[fid] = tr
                self.next_id += 1
                outputs.append((bbox, fid, tr.confirmed))
            return outputs

        # Prepare track lists
        track_ids = list(self.tracks.keys())
        track_boxes = []
        track_features = []
        # predict all tracks and get predicted boxes
        for tid in track_ids:
            tr = self.tracks[tid]
            pred = tr.predict()
            # ensure pred is in bbox form and numerical
            pred_box = [float(pred[0]), float(pred[1]), float(pred[2]), float(pred[3])]
            track_boxes.append(pred_box)
            track_features.append(tr.feature)

        # compute assignment cost and solve
        cost = self.build_cost_matrix(track_boxes, track_features, det_bboxes, det_features)
        row, col = [], []
        if cost.size > 0:
            row, col = linear_sum_assignment(cost)

        assigned_tracks = set()
        assigned_dets = set()
        outputs = []

        # apply matches with a similarity threshold
        for r, c in zip(row, col):
            cval = cost[r, c]
            sim = 1.0 - cval
            # threshold: require at least moderate similarity to accept assignment
            if sim >= 0.25:   # you can tune this (0.25 -> very permissive)
                tid = track_ids[r]
                self.tracks[tid].update(det_bboxes[c], feature=det_features[c], timestamp=timestamp)
                assigned_tracks.add(tid)
                assigned_dets.add(c)
                outputs.append((det_bboxes[c], tid, self.tracks[tid].confirmed))

        # unmatched detections -> attempt memory (reid) or create new track
        for j, bbox in enumerate(det_bboxes):
            if j in assigned_dets:
                continue
            reused_id = None
            # try to match with memory using spatial + appearance similarity
            for mid, (mbox, mfeat, mtime) in list(self.memory.items()):
                # ensure memory age not exceeded
                if timestamp - mtime > self.reid_memory_time:
                    del self.memory[mid]
                    continue
                dist = self.center_distance(bbox, mbox)
                if dist < 120:   # spatial cue for potential reid
                    app_sim = 0.0
                    if mfeat is not None and det_features[j] is not None:
                        app_sim = cosine_sim(mfeat, det_features[j])
                    if app_sim > 0.35:
                        reused_id = mid
                        del self.memory[mid]
                        break
            if reused_id is not None:
                # re-create track with same id and previous feature
                tr = ParticleTrack(bbox, reused_id, feature=det_features[j],
                                   timestamp=timestamp, num_particles=self.num_particles,
                                   confirm_hits=self.min_confirm_hits)
                tr.confirmed = tr.hits >= self.min_confirm_hits
                self.tracks[reused_id] = tr
                outputs.append((bbox, reused_id, tr.confirmed))
                assigned_dets.add(j)
                continue

            # otherwise create new track
            fid = self.next_id
            tr = ParticleTrack(bbox, fid, feature=det_features[j],
                               timestamp=timestamp, num_particles=self.num_particles,
                               confirm_hits=self.min_confirm_hits)
            tr.confirmed = tr.hits >= self.min_confirm_hits
            self.tracks[fid] = tr
            self.next_id += 1
            outputs.append((bbox, fid, tr.confirmed))
            assigned_dets.add(j)

        # update lost counters for unmatched tracks
        for tid in list(self.tracks.keys()):
            if tid not in assigned_tracks:
                tr = self.tracks[tid]
                tr.lost += 1
                # save last box & feature to memory for potential re-id with timestamp
                if tr.lost > 0:
                    self.memory[tid] = (tr.last_box, tr.feature, tr.last_seen)
                if tr.lost > self.max_lost:
                    # expire track permanently
                    if tid in self.tracks:
                        del self.tracks[tid]

        return outputs

    # small utility to get all current confirmed tracks
    def get_confirmed_tracks(self):
        return [(tr.last_box, tid) for tid, tr in self.tracks.items() if tr.confirmed]


In [6]:
# ------------------------------------------------------------
# 🔥 CELL 4 — FULL INFERENCE LOOP (YOLO EVERY FRAME)
# ------------------------------------------------------------

print("[INFO] Starting full inference (YOLO every frame)...")

results_rows = []       # final output rows for CSV
frame_buffers = {}      # track_id -> deque of cropped frames for TSF
action_predictions = {} # track_id -> (label, score)

tsf_frames_needed = CONFIG["NUM_FRAMES"]

frame_idx = 0
start_time = time.time()

while True:

    ret, frame = cap.read()
    if not ret:
        break
    if frame_idx >= total_frames:
        break

    timestamp = time.time()

    # ------------------------------------------------------
    # 1. YOLO DETECT EVERY FRAME  👈 (CHANGED)
    # ------------------------------------------------------
    yolo_out = yolo.predict(frame, conf=CONFIG["CONF_THRESH"], verbose=False)
    det_bboxes = yolo_to_bboxes(yolo_out)

    # ------------------------------------------------------
    # 2. Update tracker ALWAYS with new detections
    # ------------------------------------------------------
    tracked = tracker.update(det_bboxes, frame=frame, timestamp=timestamp)

    # Keep only confirmed tracks
    confirmed_tracks = [(bbox, tid) for (bbox, tid, conf) in tracked if conf]

    # ------------------------------------------------------
    # 3. Update frame buffers per-track
    # ------------------------------------------------------
    for (bbox, tid) in confirmed_tracks:

        if tid not in frame_buffers:
            frame_buffers[tid] = deque(maxlen=tsf_frames_needed)

        crop = crop_box(frame, bbox)
        if crop is not None and crop.size > 0:
            frame_buffers[tid].append(crop)

        # --------------------------------------------------
        # 4. Run TimeSformer if enough frames collected
        # --------------------------------------------------
        if len(frame_buffers[tid]) == tsf_frames_needed:
            clip_np = []
            for fimg in list(frame_buffers[tid]):
                resized = cv2.resize(fimg, (CONFIG["TSF_INPUT_SIZE"], CONFIG["TSF_INPUT_SIZE"]))
                clip_np.append(resized)

            clip_np = np.stack(clip_np, axis=0).astype(np.uint8)

            inputs = processor(list(clip_np), return_tensors="pt")
            inputs = {k: v.to(CONFIG["DEVICE"]) for k, v in inputs.items()}

            with torch.no_grad():
                logits = tsf_model(**inputs).logits[0]
                probs = torch.sigmoid(logits).cpu().numpy()

            top_indices = probs.argsort()[::-1][:CONFIG["TOP_K"]]
            best_idx = top_indices[0]
            best_label = action_labels[best_idx]
            best_score = float(probs[best_idx])

            # Save prediction
            action_predictions[tid] = (best_label, best_score)

    # ------------------------------------------------------
    # 5. Save results this frame
    # ------------------------------------------------------
    for (bbox, tid) in confirmed_tracks:
        x1, y1, x2, y2 = bbox

        if tid in action_predictions:
            action, score = action_predictions[tid]
        else:
            action, score = "NA", 0.0

        results_rows.append([
            frame_idx, tid,
            float(x1), float(y1), float(x2), float(y2),
            action, score
        ])

    # ------------------------------------------------------
    # 6. Optional tracker reset for out-of-frame case
    # ------------------------------------------------------
    frac_out = fraction_tracks_out_of_frame(tracker, width, height, threshold_inside=0.2)
    if frac_out > CONFIG["OUT_OF_FRAME_FRACTION"]:
        tracker = ParticleTracker(
            max_lost=30,
            iou_weight=0.5,
            dist_weight=0.2,
            app_weight=0.3,
            distance_decay=80.0,
            min_confirm_hits=3,
            reid_memory_time=CONFIG["REID_MEMORY_TIME"],
            num_particles=50
        )
        frame_buffers = {}
        action_predictions = {}
        print(f"[WARN] Tracker reset (out-of-frame fraction {frac_out:.2f})")

    frame_idx += 1

cap.release()
print("[INFO] Inference completed (YOLO every frame).")


[INFO] Starting full inference (YOLO every frame)...
[INFO] Inference completed (YOLO every frame).


In [7]:
def bbox_area_intersection_fraction(bbox, frame_w, frame_h):
    """Returns intersection area fraction relative to bbox area."""
    x1, y1, x2, y2 = bbox
    bx1, by1 = max(0, x1), max(0, y1)
    bx2, by2 = min(frame_w, x2), min(frame_h, y2)
    inter_w, inter_h = max(0, bx2 - bx1), max(0, by2 - by1)
    inter_area = inter_w * inter_h
    bbox_area = max(1.0, (x2 - x1) * (y2 - y1))
    return inter_area / bbox_area  # in [0,1]

def yolo_to_bboxes(result):
    try:
        boxes = result[0].boxes.xyxy.cpu().numpy()
    except Exception:
        boxes = result.boxes.xyxy.cpu().numpy()
    bboxes = []
    for b in boxes:
        x1, y1, x2, y2 = map(float, b[:4])
        bboxes.append([x1, y1, x2, y2])
    return bboxes

def fraction_tracks_out_of_frame(tracker, frame_w, frame_h, threshold_inside=0.2):
    confirmed = [tr for tr in tracker.tracks.values() if tr.confirmed]
    if not confirmed:
        return 0.0
    out_count = 0
    for tr in confirmed:
        x1, y1, x2, y2 = tr.last_box
        bx1, by1 = max(0, x1), max(0, y1)
        bx2, by2 = min(frame_w, x2), min(frame_h, y2)
        inter_w, inter_h = max(0, bx2 - bx1), max(0, by2 - by1)
        inter_area = inter_w * inter_h
        bbox_area = max(1.0, (x2 - x1) * (y2 - y1))
        frac = inter_area / bbox_area
        if frac < threshold_inside:
            out_count += 1
    return out_count / len(confirmed)



In [8]:
# ------------------------------------------------------------
# 🔥 CELL 4 — FULL INFERENCE LOOP (YOLO + TRACKER + TSF)
# ------------------------------------------------------------

print("[INFO] Starting full inference...")

results_rows = []       # final output rows for the CSV
frame_buffers = {}      # track_id -> deque of frames for TSF
action_predictions = {} # track_id -> last predicted (label, score)

DETECT_FREQ = CONFIG["DETECT_EVERY_N_FRAMES"]
tsf_frames_needed = CONFIG["NUM_FRAMES"]

frame_idx = 0
start_time = time.time()

while True:

    ret, frame = cap.read()
    if not ret:
        break
    if frame_idx >= total_frames:
        break

    timestamp = time.time()

    # -------------------------
    # 1. Run YOLO detections only every N frames
    # -------------------------
    if frame_idx % DETECT_FREQ == 0:
        yolo_out = yolo.predict(frame, conf=CONFIG["CONF_THRESH"], verbose=False)
        det_bboxes = yolo_to_bboxes(yolo_out)
    else:
        # No new detections = empty list triggers only TRACKER prediction
        det_bboxes = []

    # -------------------------
    # 2. Update tracker
    # -------------------------
    tracked = tracker.update(det_bboxes, frame=frame, timestamp=timestamp)

    # tracked = list of (bbox, track_id, confirmed_bool)
    confirmed_tracks = [(bbox, tid) for (bbox, tid, conf) in tracked if conf]

    # -------------------------
    # 3. For each confirmed track, maintain frame buffers
    # -------------------------
    for (bbox, tid) in confirmed_tracks:

        # Init deque for each new track
        if tid not in frame_buffers:
            frame_buffers[tid] = deque(maxlen=tsf_frames_needed)

        # Append current cropped person ROI into the clip buffer
        crop = crop_box(frame, bbox)
        if crop is not None and crop.size > 0:
            frame_buffers[tid].append(crop)

        # -------------------------
        # 4. Run TimeSformer if we have enough frames
        # -------------------------
        if len(frame_buffers[tid]) == tsf_frames_needed:
            # Preprocess: resize + stack
            clip_np = []
            for fimg in list(frame_buffers[tid]):
                resized = cv2.resize(fimg, (CONFIG["TSF_INPUT_SIZE"], CONFIG["TSF_INPUT_SIZE"]))
                clip_np.append(resized)

            clip_np = np.stack(clip_np, axis=0)  # (T, H, W, C)
            clip_np = clip_np.astype(np.uint8)

            # Processor expects dict
            inputs = processor(list(clip_np), return_tensors="pt")
            inputs = {k: v.to(CONFIG["DEVICE"]) for k, v in inputs.items()}

            with torch.no_grad():
                logits = tsf_model(**inputs).logits[0]
                probs = torch.sigmoid(logits).cpu().numpy()

            # pick top-K actions
            top_indices = probs.argsort()[::-1][:CONFIG["TOP_K"]]
            best_idx = top_indices[0]
            best_label = action_labels[best_idx]
            best_score = float(probs[best_idx])

            # save last action prediction
            action_predictions[tid] = (best_label, best_score)

    # -------------------------
    # 5. Save results this frame for all tracks
    # -------------------------
    for (bbox, tid) in confirmed_tracks:
        x1, y1, x2, y2 = bbox

        if tid in action_predictions:
            action, score = action_predictions[tid]
        else:
            action, score = "NA", 0.0

        results_rows.append([
            frame_idx, tid,
            float(x1), float(y1), float(x2), float(y2),
            action, score
        ])

    # -------------------------
    # 6. Out-of-frame reset check (optional)
    # -------------------------
    frac_out = fraction_tracks_out_of_frame(tracker, width, height, threshold_inside=0.2)
    if frac_out > CONFIG["OUT_OF_FRAME_FRACTION"]:
        tracker = ParticleTracker(
            max_lost=30,
            iou_weight=0.5,
            dist_weight=0.2,
            app_weight=0.3,
            distance_decay=80.0,
            min_confirm_hits=3,
            reid_memory_time=CONFIG["REID_MEMORY_TIME"],
            num_particles=50
        )
        frame_buffers = {}
        action_predictions = {}
        print(f"[WARN] Tracker reset (out-of-frame fraction {frac_out:.2f})")

    frame_idx += 1

cap.release()
print("[INFO] Inference completed.")


[INFO] Starting full inference...
[INFO] Inference completed.


In [9]:
# ---------------------------
# Cell 5: Save inference results to CSV
# ---------------------------
import pandas as pd
cols = ["frame", "track_id", "xmin", "ymin", "xmax", "ymax", "action", "score"]
pred_df = pd.DataFrame(results_rows, columns=cols)

# Save CSV
pred_csv_path = CONFIG.get("RESULTS_CSV", f"{CONFIG['OUTPUT_DIR']}/{CONFIG['VIDEO_NAME']}_predictions.csv")
pred_df.to_csv(pred_csv_path, index=False)
print(f"[INFO] Saved predictions to: {pred_csv_path}")


[INFO] Saved predictions to: /home1/jtt_1/mtp/outputs/2.1.8/2.1.8_tracking.csv
